# Imports

In [49]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

import pyarrow
import matplotlib.pyplot as plt
import re
import pandas as pd
import os

In [31]:
data=pd.read_csv("data/raw/buildstock/MN_upgrade0.csv", dtype={'MN_upgrade0.csv': 'str'})

/tmp/ipykernel_7755/105414082.py:1: DtypeWarning: Columns (5,144,151,152,359,360) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv("data/raw/buildstock/MN_upgrade0.csv", dtype={'MN_upgrade0.csv': 'str'})


# Constants
Daddy's made you some constants.

In [32]:
def get_parquet_file_path(state_abbreviation: str, building_id: str) -> str:
    """Returns the directory path for parquet files based on the state abbreviation."""
    return f"data/raw/nrel_parquet/{state_abbreviation.lower()}/{building_id}-0.parquet"

columns_of_interest = [
    "MN_upgrade0.csv",
    "in.city",
    "in.sqft..ft2",
    'in.heating_fuel',
    'in.heating_setpoint',
    "in.insulation_ceiling",
    "in.insulation_floor",
    "in.insulation_foundation_wall",
    "in.insulation_rim_joist",
    "in.insulation_roof",
    "in.geometry_building_type_recs",
    "out.params.size_heat_pump_backup_primary..kbtu_per_hr",
    "out.params.size_heating_system_primary..kbtu_per_hr",
    "out.params.size_heating_system_secondary..kbtu_per_hr"
]

column_renames = {
    "MN_upgrade0.csv": "building_id",
    "in.city": "city",
    "in.sqft..ft2": 'area',
    "in.heating_fuel": 'heating_fuel',
    "in.heating_setpoint": 'heating_setpoint',
    "in.insulation_ceiling": 'ceiling_r',
    "in.insulation_floor": 'floor_r',
    "in.insulation_foundation_wall": 'foundation_wall_r',
    "in.insulation_rim_joist": 'rim_joist_r',
    "in.insulation_roof": 'roof_r',
    "in.geometry_building_type_recs": 'building_type',
    "out.params.size_heat_pump_backup_primary..kbtu_per_hr": 'heat_pump_backup_primary_size',
    "out.params.size_heating_system_primary..kbtu_per_hr": 'heating_system_primary_size',
    "out.params.size_heating_system_secondary..kbtu_per_hr": 'heating_system_secondary_size'
}

df = data[columns_of_interest].copy()
df.rename(columns=column_renames, inplace=True)
df['building_id'] = df['building_id'].astype('str')

# filter file down to only the columns of interest and only the rows for Minneapolis, MN
df = df.where(
    (df["city"] == "MN, Minneapolis")
    & (df["building_type"] == "Single-Family Detached" ), axis=0
).dropna()

In [34]:
df['building_id'].unique()

array(['1641', '6250', '9572', '13087', '13786', '22043', '27407',
       '30507', '31874', '32331', '35520', '38425', '40952', '45931',
       '47534', '57625', '65224', '67464', '72940', '81997', '82186',
       '82862', '83178', '86765', '90349', '92004', '94083', '98308',
       '98992', '100745', '101421', '103386', '103928', '105194',
       '108104', '111805', '125109', '125389', '133093', '138909',
       '153329', '153974', '157454', '159647', '161029', '161714',
       '165633', '167877', '171825', '173238', '179657', '181617',
       '181832', '182925', '190720', '191361', '198065', '200457',
       '205050', '205881', '209362', '213254', '213931', '223019',
       '227783', '231096', '234796', '235665', '235990', '236632',
       '237987', '240687', '241727', '245316', '252437', '253128',
       '253632', '253964', '254216', '254612', '257267', '261019',
       '268305', '269531', '272747', '272790', '274938', '276276',
       '281120', '283116', '284960', '286536', '292265

In [ ]:
# Standard baseline R-values for uninsulated (unfinished) building elements
UNFINISHED_R_VALUES = {
    "floor_r": 2.0,  # 4" concrete slab + ground contact + air film
    "foundation_wall_r": 2.0,  # 8" poured concrete wall + air films
    "ceiling_r": 2.5,  # 5/8" drywall/joists + still air film
    "rim_joist_r": 2.5,  # 1.5" wood rim board + sheathing + air films
    "roof_r": 3.0,  # 5/8" OSB decking + asphalt shingles + air films
}

def parse_r_value(val: str, default_r: float, col_name: str = "not specified") -> float:
    """Extracts the numeric float directly following 'r-' or 'R-'.
    Defaults to default_r if 'r-<number>' isn't present or val is missing.
    """
    if pd.isna(val):
        return default_r

    # Looks specifically for 'r-' or 'R-' followed by digits/decimals
    match = re.search(r'r-(\d+(?:\.\d+)?)', str(val), flags=re.IGNORECASE)

    return float(match.group(1)) if match else default_r


def calculate_unified_r(
    df: pd.DataFrame, area_weights: dict = None
) -> pd.DataFrame:
    """Parses string R-values to floats and computes a unified parallel envelope R-value."""
    df_out = df.copy()
    r_cols = list(UNFINISHED_R_VALUES.keys())
    num_cols = []

    # Parse columns into numeric floats
    for col in r_cols:
        num_col = f"{col}_float"
        default = UNFINISHED_R_VALUES[col]
        df_out[num_col] = df_out[col].apply(
            lambda x: parse_r_value(x, default, col_name=col)
        )
        num_cols.append(num_col)

    # Parallel Heat Flow Calculation: U_total = sum(U_i) -> R_unified = 1 / U_avg
    u_values = 1.0 / df_out[num_cols].values

    if area_weights:
        # Area-weighted harmonic mean: R_unified = sum(A_i) / sum(A_i / R_i)
        weights = np.array([area_weights[col] for col in r_cols])
        df_out["unified_r_value"] = weights.sum() / (u_values @ weights)
    else:
        # Unweighted harmonic mean across the 5 assemblies
        df_out["unified_r_value"] = len(r_cols) / u_values.sum(axis=1)

    return df_out


df['ceiling_r'].unique()


df = calculate_unified_r(df)

In [19]:
df['building_type'].unique()

array(['Single-Family Detached', 'Single-Family Attached'], dtype=object)

In [84]:
def find_closest_row(df, heating_fuel, heating_setpoint, unified_r, area):
    numeric_columns = ["heating_setpoint", "unified_r_value", "area"]
    numeric_values = df[numeric_columns].copy()
    numeric_values["heating_setpoint"] = pd.to_numeric(
        numeric_values["heating_setpoint"]
        .astype("string")
        .str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False),
        errors="coerce",
    )
    numeric_values[["unified_r_value", "area"]] = numeric_values[
        ["unified_r_value", "area"]
    ].apply(pd.to_numeric, errors="coerce")

    candidates = df[df["heating_fuel"] == heating_fuel]
    if candidates.empty:
        raise ValueError(f"No rows found for heating_fuel={heating_fuel!r}")

    candidate_values = numeric_values.loc[candidates.index].dropna()
    if candidate_values.empty:
        raise ValueError(f"No numeric rows found for heating_fuel={heating_fuel!r}")

    setpoint = pd.to_numeric(
        pd.Series([heating_setpoint], dtype="string").str.extract(
            r"([-+]?\d+(?:\.\d+)?)", expand=False
        ).iloc[0],
        errors="coerce",
    )
    target = pd.Series([setpoint, unified_r, area], index=numeric_columns, dtype=float)
    scales = numeric_values.std().replace(0, 1)
    distances = (((candidate_values - target) / scales) ** 2).sum(axis=1)
    return df.loc[distances.idxmin()]


In [88]:
find_closest_row(df, heating_fuel="Natural Gas", heating_setpoint=70, unified_r=15, area=2000)

bldg_id                                         274938.0
city                                     MN, Minneapolis
area                                              3310.0
heating_fuel                                 Natural Gas
heating_setpoint                                     68F
ceiling_r                                           R-49
floor_r                                      Uninsulated
foundation_wall_r                    Wall R-15, Exterior
rim_joist_r                               R-15, Exterior
roof_r                           Unfinished, Uninsulated
heat_pump_backup_primary_size                        0.0
heating_system_primary_size                        90.93
heating_system_secondary_size                        0.0
floor_r_float                                        2.0
foundation_wall_r_float                             15.0
ceiling_r_float                                     49.0
rim_joist_r_float                                   15.0
roof_r_float                   

In [39]:
# os.path.isfile(get_parquet_file_path("MN", '274938'))


True

In [43]:
df_ts = pd.read_parquet(get_parquet_file_path("MN", '274938'))

cols = list(df_ts.columns)
cols.sort()
cols

# The column we care about is:
'out.load.heating.energy_delivered..kbtu'

['bldg_id',
 'in.sqft',
 'out.electricity.ceiling_fan.energy_consumption..kwh',
 'out.electricity.ceiling_fan.energy_consumption_intensity..kwh_per_ft2',
 'out.electricity.clothes_dryer.energy_consumption..kwh',
 'out.electricity.clothes_dryer.energy_consumption_intensity..kwh_per_ft2',
 'out.electricity.clothes_washer.energy_consumption..kwh',
 'out.electricity.clothes_washer.energy_consumption_intensity..kwh_per_ft2',
 'out.electricity.cooling.energy_consumption..kwh',
 'out.electricity.cooling.energy_consumption_intensity..kwh_per_ft2',
 'out.electricity.cooling_fans_pumps.energy_consumption..kwh',
 'out.electricity.cooling_fans_pumps.energy_consumption_intensity..kwh_per_ft2',
 'out.electricity.dishwasher.energy_consumption..kwh',
 'out.electricity.dishwasher.energy_consumption_intensity..kwh_per_ft2',
 'out.electricity.ev_charging.energy_consumption..kwh',
 'out.electricity.ev_charging.energy_consumption_intensity..kwh_per_ft2',
 'out.electricity.freezer.energy_consumption..kwh',


In [46]:
df_temp = pd.read_csv("./data/raw/weather/mn.csv")

In [47]:
list(df_temp.columns)

['date_time',
 'Dry Bulb Temperature [°C]',
 'Relative Humidity [%]',
 'Wind Speed [m/s]',
 'Wind Direction [Deg]',
 'Global Horizontal Radiation [W/m2]',
 'Direct Normal Radiation [W/m2]',
 'Diffuse Horizontal Radiation [W/m2]']

# Model

In [50]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d


def run_thermal_hp_sim(
    df_hourly: pd.DataFrame,
    area_sqft: float = 3310.0,
    r_val: float = 5.065,
    setpoint_f: float = 68.0,
    deadband_f: float = 1.0,
    # Standard multi-point COP ratings dict: {temp_f: COP}
    cop_ratings: dict[float, float] | None = None,
    # Standard multi-point capacity ratings dict: {temp_f: max_btu_per_hr}
    capacity_ratings: dict[float, float] | None = None,
    hp_capacity_tons: float = 3.0,  # Used as fallback baseline if capacity_ratings is None
    gas_furnace_eff: float = 0.80,
    elec_rate_per_kwh: float = 0.15,
    gas_rate_per_therm: float = 1.20,
    grid_co2_kg_per_kwh: float = 0.45,
    gas_co2_kg_per_therm: float = 5.30,
) -> tuple[pd.DataFrame, dict]:
    """Runs an hourly 1-Node RC thermal simulation for a heat pump home with gas backup,

    using 1D linear interpolation for cold-climate COP and capacity curves.
    """
    # 1. Provide default NEEP/AHRI benchmark rating curves if none supplied
    if cop_ratings is None:
        # Standard cold-climate inverter profile (47°F, 17°F, 5°F, -22°F)
        cop_ratings = {47.0: 3.60, 17.0: 2.40, 5.0: 1.80, -22.0: 1.00}

    if capacity_ratings is None:
        # Base 3-ton unit (36,000 BTU/hr nominal at 47°F) with cold weather capacity drop
        capacity_ratings = {
            47.0: hp_capacity_tons * 12000.0,
            17.0: hp_capacity_tons * 12000.0 * 0.78,
            5.0: hp_capacity_tons * 12000.0 * 0.60,
            -22.0: hp_capacity_tons * 12000.0 * 0.30,
        }

    # Sort dictionary keys to ensure ascending arrays for interpolation
    temps_cop = np.array(sorted(cop_ratings.keys()))
    vals_cop = np.array([cop_ratings[t] for t in temps_cop])

    temps_cap = np.array(sorted(capacity_ratings.keys()))
    vals_cap = np.array([capacity_ratings[t] for t in temps_cap])

    # 2. Build SciPy linear interpolation functions
    get_cop_interp = interp1d(
        temps_cop,
        vals_cop,
        kind='linear',
        bounds_error=False,
        fill_value=(vals_cop[0], vals_cop[-1]),  # Clip at boundaries
    )

    get_cap_interp = interp1d(
        temps_cap,
        vals_cap,
        kind='linear',
        bounds_error=False,
        fill_value=(vals_cap[0], vals_cap[-1]),  # Clip at boundaries
    )

    # 3. Setup Building & Simulation State
    n_hours = len(df_hourly)
    ua_btu = area_sqft / r_val
    c_btu = area_sqft * 15.0

    t_in = np.zeros(n_hours)
    q_hp_delivered = np.zeros(n_hours)
    q_aux_delivered = np.zeros(n_hours)
    kwh_hp = np.zeros(n_hours)
    therms_aux = np.zeros(n_hours)
    hvac_action = ['OFF'] * n_hours

    t_in[0] = setpoint_f
    dt_hr = 1.0

    # 4. Hourly Simulation Loop
    for t in range(n_hours - 1):
        t_amb = df_hourly['t_amb_f'].iloc[t]
        q_int = df_hourly['q_int_btu'].iloc[t]

        heat_loss = ua_btu * (t_in[t] - t_amb)
        heat_needed = max(0.0, heat_loss - q_int)

        if t_in[t] < (setpoint_f - deadband_f) or heat_needed > 0:
            # Look up interpolated COP and max capacity for current outdoor temp
            cop_current = float(get_cop_interp(t_amb))
            avail_hp_cap = float(get_cap_interp(t_amb))

            # Heat delivered split
            hp_heat = min(heat_needed, avail_hp_cap)
            aux_heat = max(0.0, heat_needed - hp_heat)

            # Energy inputs
            hp_elec_kwh = (hp_heat / max(1.0, cop_current)) / 3412.14
            gas_therms = (aux_heat / gas_furnace_eff) / 100000.0

            if hp_heat > 0 and aux_heat > 0:
                hvac_action[t] = 'HP_AND_GAS_AUX'
            elif hp_heat > 0:
                hvac_action[t] = 'HP_ONLY'
            elif aux_heat > 0:
                hvac_action[t] = 'GAS_AUX_ONLY'
        else:
            hp_heat = 0.0
            aux_heat = 0.0
            hp_elec_kwh = 0.0
            gas_therms = 0.0
            hvac_action[t] = 'OFF'

        q_hp_delivered[t] = hp_heat
        q_aux_delivered[t] = aux_heat
        kwh_hp[t] = hp_elec_kwh
        therms_aux[t] = gas_therms

        # State update
        net_heat_gain = hp_heat + aux_heat + q_int - heat_loss
        t_in[t + 1] = t_in[t] + (net_heat_gain / c_btu) * dt_hr

    hvac_action[-1] = hvac_action[-2]

    # 5. Format Output Dataframe
    df_res = df_hourly.copy()
    df_res['t_in_f'] = t_in
    df_res['hvac_action'] = hvac_action
    df_res['q_hp_btu'] = q_hp_delivered
    df_res['q_aux_btu'] = q_aux_delivered
    df_res['kwh_hp'] = kwh_hp
    df_res['therms_aux'] = therms_aux

    df_res['elec_cost_usd'] = df_res['kwh_hp'] * elec_rate_per_kwh
    df_res['gas_cost_usd'] = df_res['therms_aux'] * gas_rate_per_therm
    df_res['total_cost_usd'] = (
        df_res['elec_cost_usd'] + df_res['gas_cost_usd']
    )

    df_res['co2_elec_kg'] = df_res['kwh_hp'] * grid_co2_kg_per_kwh
    df_res['co2_gas_kg'] = df_res['therms_aux'] * gas_co2_kg_per_therm
    df_res['total_co2_kg'] = df_res['co2_elec_kg'] + df_res['co2_gas_kg']

    summary = {
        'total_elec_kwh': df_res['kwh_hp'].sum(),
        'total_gas_therms': df_res['therms_aux'].sum(),
        'total_cost_usd': df_res['total_cost_usd'].sum(),
        'total_co2_kg': df_res['total_co2_kg'].sum(),
        'hp_coverage_pct': (
            df_res['q_hp_btu'].sum()
            / max(1.0, (df_res['q_hp_btu'].sum() + df_res['q_aux_btu'].sum()))
        )
        * 100,
        'min_indoor_temp_f': df_res['t_in_f'].min(),
    }

    return df_res, summary


# --- Sanity Check with Custom Rating Curves ---
if __name__ == '__main__':
    hours = pd.date_range('2026-01-01', periods=8760, freq='h')
    np.random.seed(42)
    mock_t_amb = 40.0 - 35.0 * np.cos(
        2 * np.pi * np.arange(8760) / 8760
    ) + np.random.normal(0, 4, 8760)
    mock_q_int = np.random.uniform(1200, 2800, 8760)

    df_input = pd.DataFrame(
        {'t_amb_f': mock_t_amb, 'q_int_btu': mock_q_int}, index=hours
    )

    # Custom specifications for a cold-climate heat pump
    user_cops = {47.0: 3.80, 17.0: 2.60, 5.0: 2.00, -22.0: 1.15}
    user_caps = {
        47.0: 36000.0,
        17.0: 32000.0,
        5.0: 27000.0,
        -22.0: 14000.0,
    }  # Max capacity in BTU/hr

    results_df, summary = run_thermal_hp_sim(
        df_hourly=df_input,
        cop_ratings=user_cops,
        capacity_ratings=user_caps,
        elec_rate_per_kwh=0.17,
        gas_rate_per_therm=1.20,
    )

    print('=== Annual Summary ===')
    for k, v in summary.items():
        print(f'{k}: {v:,.2f}')

    print('\n=== Sample Hours Sanity Check ===')
    print(
        results_df[
            [
                't_amb_f',
                't_in_f',
                'hvac_action',
                'q_hp_btu',
                'q_aux_btu',
                'kwh_hp',
                'therms_aux',
            ]
        ].head(10)
    )

=== Annual Summary ===
total_elec_kwh: 17,233.14
total_gas_therms: 414.47
total_cost_usd: 3,082.34
total_co2_kg: 9,951.62
hp_coverage_pct: 82.71
min_indoor_temp_f: 68.00

=== Sample Hours Sanity Check ===
                       t_amb_f  t_in_f     hvac_action      q_hp_btu  \
2026-01-01 00:00:00   6.986857    68.0  HP_AND_GAS_AUX  27827.856922   
2026-01-01 01:00:00   4.446952    68.0  HP_AND_GAS_AUX  26733.717533   
2026-01-01 02:00:00   7.590790    68.0  HP_AND_GAS_AUX  28079.495902   
2026-01-01 03:00:00  11.092200    68.0  HP_AND_GAS_AUX  29538.416855   
2026-01-01 04:00:00   4.063531    68.0  HP_AND_GAS_AUX  26549.107302   
2026-01-01 05:00:00   4.063677    68.0  HP_AND_GAS_AUX  26549.177934   
2026-01-01 06:00:00  11.317175    68.0  HP_AND_GAS_AUX  29632.156405   
2026-01-01 07:00:00   8.070180    68.0  HP_AND_GAS_AUX  28279.241694   
2026-01-01 08:00:00   3.122679    68.0  HP_AND_GAS_AUX  26096.104535   
2026-01-01 09:00:00   7.170969    68.0  HP_AND_GAS_AUX  27904.570591   

  

# Parameter search

In [53]:
np.random.seed(42)
hours = pd.date_range("2026-01-01", periods=8760, freq="h")
mock_t_amb = 40.0 - 35.0 * np.cos(
    2 * np.pi * np.arange(8760) / 8760
) + np.random.normal(0, 4, 8760)
mock_q_int = np.random.uniform(1200, 2800, 8760)
df_input = pd.DataFrame({"t_amb_f": mock_t_amb, "q_int_btu": mock_q_int}, index=hours)

# 2. Define search parameters to evaluate
capacity_options_tons = [0.0, 2.5, 3.5, 4.5]

# Different heat pump performance tiers (Standard vs. Cold Climate Premium)
cop_profiles = {
    "Standard_ccASHP": {47.0: 3.40, 17.0: 2.20, 5.0: 1.50, -22.0: 1.00},
    "Premium_ccASHP": {47.0: 4.10, 17.0: 2.80, 5.0: 2.20, -22.0: 1.30},
}

results_list = []

# 3. Parameter grid search execution loop
for profile_name, cop_dict in cop_profiles.items():
    for tons in capacity_options_tons:
        # Scale capacity curve based on tonnage
        cap_dict = {
            47.0: tons * 12000.0,
            17.0: tons * 12000.0 * 0.80,
            5.0: tons * 12000.0 * 0.65,
            -22.0: tons * 12000.0 * 0.35,
        }

        # Run simulation run
        _, summary = run_thermal_hp_sim(
            df_hourly=df_input,
            cop_ratings=cop_dict,
            capacity_ratings=cap_dict,
            hp_capacity_tons=tons,
            elec_rate_per_kwh=0.15, # 1 kWh is 0.0341 therms, so $0.15/kWh is equivalent to $4.40/therm
            gas_rate_per_therm=1.20,
        )

        # Log search metadata
        summary["profile"] = profile_name
        summary["hp_size_tons"] = tons
        results_list.append(summary)

# 4. Compile and format summary results DataFrame
df_summary = pd.DataFrame(results_list)

# Organize and display table
columns_order = [
    "profile",
    "hp_size_tons",
    "hp_coverage_pct",
    "total_elec_kwh",
    "total_gas_therms",
    "total_cost_usd",
    "total_co2_kg",
    "min_indoor_temp_f",
]
df_summary = df_summary[columns_order].sort_values(by="hp_size_tons")

print("\n=== HEAT PUMP SIZING & COP SEARCH MATRIX ===")
print(df_summary.to_string(index=False))


=== HEAT PUMP SIZING & COP SEARCH MATRIX ===
        profile  hp_size_tons  hp_coverage_pct  total_elec_kwh  total_gas_therms  total_cost_usd  total_co2_kg  min_indoor_temp_f
Standard_ccASHP           0.0         0.000000        0.000000       2397.809968     2877.371961  12708.392829               68.0
 Premium_ccASHP           0.0         0.000000        0.000000       2397.809968     2877.371961  12708.392829               68.0
 Premium_ccASHP           2.5        69.821448    13035.811467        723.624327     2823.720913   9701.324094               68.0
Standard_ccASHP           2.5        69.821448    16967.503557        723.624327     3413.474726  11470.585535               68.0
 Premium_ccASHP           3.5        84.148524    16187.204996        380.088274     2884.186679   9298.710102               68.0
Standard_ccASHP           3.5        84.148524    21232.000702        380.088274     3640.906034  11568.868169               68.0
Standard_ccASHP           4.5        93.2742